# 🚀 Inventory Agent Fine-Tuning in Google Colab

### 1. Mount Google Drive
Run this cell and authenticate to give Colab access to your datasets and to save the trained model checkpoints directly back to your Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install LLaMA-Factory & Dependencies

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e ".[torch,metrics]"
!pip install bitsandbytes

### 3. Setup Datasets
We will copy the dataset files from your Drive into LLaMA-Factory and register them in `dataset_info.json`.
*(Note: Make sure `DRIVE_PATH` matches exactly where your inventory folder is located in your Drive!)*

In [ ]:
import os
import shutil
import json

# ==========================================
# CHANGE THIS TO YOUR EXACT INVENTORY FOLDER PATH IN DRIVE
DRIVE_PATH = "/content/drive/MyDrive/inventory"
# ==========================================

os.makedirs("data", exist_ok=True)

# Copy the generated datasets
shutil.copy(f"{DRIVE_PATH}/finetune/text_to_sql_alpaca.jsonl", "data/text_to_sql_alpaca.jsonl")
shutil.copy(f"{DRIVE_PATH}/finetune/tool_calling_chatml.jsonl", "data/tool_calling_chatml.jsonl")

# Load existing dataset_info and append our custom datasets
with open("data/dataset_info.json", "r") as f:
    dataset_info = json.load(f)

dataset_info["inventory_text_to_sql"] = {
  "file_name": "text_to_sql_alpaca.jsonl",
  "formatting": "alpaca"
}
dataset_info["inventory_tool_calling"] = {
  "file_name": "tool_calling_chatml.jsonl",
  "formatting": "sharegpt",
  "columns": {"messages": "messages"},
  "tags": {
    "role_tag": "role",
    "content_tag": "content",
    "user_role": "user",
    "assistant_role": "assistant"
  }
}

with open("data/dataset_info.json", "w") as f:
    json.dump(dataset_info, f, indent=2)

print("Datasets registered successfully!")

### 4. Create Training Config
We'll use 4-bit QLoRA to fine-tune the `Qwen/Qwen2.5-7B-Instruct` model efficiently on a free T4 GPU (or A100 if you have Colab Pro).

This cell writes the configuration to `text_to_sql.yaml`.

In [ ]:
%%writefile text_to_sql.yaml
model_name_or_path: Qwen/Qwen2.5-7B-Instruct
stage: sft
do_train: true
finetuning_type: lora
lora_target: all
lora_rank: 8

dataset: inventory_text_to_sql
template: qwen
cutoff_len: 2048

# Saves the model directly back to your Google Drive!
output_dir: /content/drive/MyDrive/inventory/finetune/checkpoints/text_to_sql
logging_steps: 10
save_steps: 100
plot_loss: true
overwrite_output_dir: true

per_device_train_batch_size: 2
gradient_accumulation_steps: 8
learning_rate: 2.0e-5
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.1

# Enable fp16 (T4 GPU compatible) + 4-bit quantization
fp16: true
quantization_bit: 4


### 5. Start Training
Execute the training script. Expect this to take roughly **15–30 minutes**. Once finished, you will find the LoRA weights in your Google Drive.

In [ ]:
!llamafactory-cli train text_to_sql.yaml